<a href="https://colab.research.google.com/github/nina-bilirossi/Masters_thesis/blob/main/data_retrieval_copernicus_thesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ERA5 Satelite data retrieval (Nina's master's thesis)

I ran this code in google colab, which allows smooth connection with GEE and saves the documents to your google drive. For any questions you can reach me at nina.bilirossi@gmail.com.

*AI disclosure* : the code was mostly generated using AI given my instructions. I then checked it for coherence and accuracy.

## Set up

To use the code, you need to create a Google Earth Engine (GEE) project and replace the name `aqui-460614` to your project name.

Once you have created your project, you can import the shapefiles for India (Subdistrict_boundaries and District_boundaries) into the GEE code editor (https://code.earthengine.google.com/). I named them "Sub-district_Boundary" and "districts", and this is how they are referred to in the code below. You can find the shaepfiles on the GitHub repository under `data > geography`.

Note that it takes some time to upload the shapefiles to GEE, so do not be concerned if it does not appear in your assets right away.

You will need to grant access to your Google Drive. Run the cell bellow and check the boxes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Precipitation data

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# Export total daily precipitation for each year and subdistrict
# ══════════════════════════════════════════════════════════════════════════════

# This code retrieves the maximum daily precipitation within each subdistrict for every day between 1994 and 2024
# the subdistrict value is defined as the maximum over the pixels in that subdistrict (capturing extremes)

# Using subdistricts (34/36 states)
# 2 states do not have subdistrict boundaries defined, we deal with them later.

import ee
ee.Authenticate()

# 1. Initialize
ee.Initialize(project='aqui-460614')

# 2. Define Inputs
asset_path = "projects/aqui-460614/assets/Sub-district_Boundary"
subdistricts = ee.FeatureCollection(asset_path)

# OPTIMIZATION: Simplify boundaries (100m tolerance)
# This makes the spatial math much faster without losing 9km-grid accuracy.
subdistricts_simple = subdistricts.map(lambda f: f.simplify(maxError=100))

#print(subdistricts.size().getInfo())         # expect 34 (2 states have no subdistrict granularity in the shapefile)

start_year = 1994
end_year = 2025
district_name_col = "SUB_DIST"

def export_year_fast(year):
    # Load and scale to mm
    era5_col = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
                .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
                .select('total_precipitation_sum'))

    # OPTIMIZATION: Convert Collection to a single Image with 365 Bands
    # We rename each band to the date so the CSV headers are clean
    def rename_bands(img):
        date_str = img.date().format("YYYY_MM_DD")
        return img.multiply(1000).rename(date_str)

    multi_band_image = era5_col.map(rename_bands).toBands()

    # One single spatial reduction for the entire year
    stats = multi_band_image.reduceRegions(
        collection=subdistricts_simple,
        reducer=ee.Reducer.max(),
        scale=11132
    )

    # Export to Drive
    task = ee.batch.Export.table.toDrive(
        collection=stats,
        description=f"MAX_Precip_Wide_{year}",
        folder="ERA5_MAX_Precipitation_Optimized",
        fileFormat="CSV"
    )
    task.start()
    print(f"Submitted MAX optimized task for {year}")

In [10]:
# Run the loop
for y in range(start_year, end_year + 1):
    export_year_fast(y)

Submitted MAX optimized task for 1994
Submitted MAX optimized task for 1995


You can check that the tasks are running on https://code.earthengine.google.com/. You can also see the task progress there, including when they are completed and available to download from your Google Drive.

The code below allows to complete our precipitation data with the two missing states, using district level max precipitation values rather than subdistrict.

In [ ]:
# PRECIPITATION DOWNLOADS
# Only for the two states without subdistricts

import ee
ee.Authenticate()
ee.Initialize(project='aqui-460614')

# 1. Load DISTRICT shapefile (replace path if needed)
districts = ee.FeatureCollection("projects/aqui-460614/assets/districts")

# 2. Filter ONLY required states FIRST (important for speed)
target_states = ['ARUNACHAL PRADESH', 'MEGHALAYA']

districts_filtered = districts.filter(
    ee.Filter.inList('STATE_UT', target_states)
)

# Optional geometry simplification
districts_simple = districts_filtered.map(
    lambda f: f.simplify(maxError=100)
)

start_year = 1994
end_year = 2024

def export_year_fast(year):

    era5_col = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
                .filterDate(f"{year}-01-01", f"{year+1}-01-01")
                .select('total_precipitation_sum'))

    # Rename bands to date
    def rename_bands(img):
        return img.multiply(1000).rename(
            img.date().format("YYYY_MM_DD")
        )

    multi_band = era5_col.map(rename_bands).toBands()

    # Reduce over districts (already filtered)
    stats = multi_band.reduceRegions(
        collection=districts_simple,
        reducer=ee.Reducer.max(),
        scale=11132
    )

    # print(f"Features for {year}:", stats.size().getInfo())

    # Export
    task = ee.batch.Export.table.toDrive(
        collection=stats,
        description=f"MAX_Precip_Districts_{year}",
        folder="ERA5_MAX_Precip_Districts",
        fileFormat="CSV"
    )
    task.start()

    # print(f"Submitted task for {year}")

# Run loop
for year in range(start_year, end_year + 1):
    export_year_fast(year)


## Soil Moisture Antecedent

In [14]:
# ── GEE setup ─────────────────────────────────────────────────────────────────
ee.Authenticate()
ee.Initialize(project='aqui-460614')

ASSET_PATH      = "projects/aqui-460614/assets/Sub-district_Boundary"
DISTRICT_COL    = "SUB_DIST"
DRIVE_FOLDER    = "ERA5_SoilMoisture_Raw"
START_YEAR      = 1994
END_YEAR        = 2024
SCALE           = 11132   # ~ERA5-Land native resolution in metres

In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# Export raw daily soil moisture for each year
# ══════════════════════════════════════════════════════════════════════════════

def export_raw_daily(year: int):
    """
    Exports a wide CSV for `year`:
      columns = subdistrict attributes + one column per day (YYYY_MM_DD)
      values  = mean volumetric_soil_water_layer_1 (m³/m³) per subdistrict
    """
    subdistricts = ee.FeatureCollection(ASSET_PATH)
    subdistricts_simple = subdistricts.map(lambda f: f.simplify(maxError=100))

    col = (
        ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
        .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
        .select("volumetric_soil_water_layer_1")
    )

    def rename_band(img):
        date_str = img.date().format("YYYY_MM_DD")
        return img.rename(date_str)

    multi_band = col.map(rename_band).toBands()

    stats = multi_band.reduceRegions(
        collection=subdistricts_simple,
        reducer=ee.Reducer.mean(),
        scale=SCALE,
    )

    task = ee.batch.Export.table.toDrive(
        collection=stats,
        description=f"SM_Raw_Daily_{year}",
        folder=DRIVE_FOLDER,
        fileFormat="CSV",
    )
    task.start()
    print(f"  ✓ Submitted raw daily export for {year}")

In [16]:
print("=== Submitting GEE export tasks ===")
for y in range(START_YEAR, END_YEAR + 1):
    export_raw_daily(y)
print("All tasks submitted.  Download the CSVs from Google Drive.")


=== Submitting GEE export tasks ===
  ✓ Submitted raw daily export for 1994
  ✓ Submitted raw daily export for 1995
  ✓ Submitted raw daily export for 1996
  ✓ Submitted raw daily export for 1997
  ✓ Submitted raw daily export for 1998
  ✓ Submitted raw daily export for 1999
  ✓ Submitted raw daily export for 2000
  ✓ Submitted raw daily export for 2001
  ✓ Submitted raw daily export for 2002
  ✓ Submitted raw daily export for 2003
  ✓ Submitted raw daily export for 2004
  ✓ Submitted raw daily export for 2005
  ✓ Submitted raw daily export for 2006
  ✓ Submitted raw daily export for 2007
  ✓ Submitted raw daily export for 2008
  ✓ Submitted raw daily export for 2009
  ✓ Submitted raw daily export for 2010
  ✓ Submitted raw daily export for 2011
  ✓ Submitted raw daily export for 2012
  ✓ Submitted raw daily export for 2013
  ✓ Submitted raw daily export for 2014
  ✓ Submitted raw daily export for 2015
  ✓ Submitted raw daily export for 2016
  ✓ Submitted raw daily export for 2017
  ✓ 

You can check that the tasks are running on https://code.earthengine.google.com/. You can also see the task progress there, including when they are completed and available to download from your Google Drive.